# 00 - Naive RAG and naive Q&A generation

Steps in this notebook:

1. Setup and model access check
2. Document preparation
3. Chunking
4. Indexing
5. Naive RAG
6. Naive Q&A generation
7. First evaluation

See [README.md](./README.md) for the full design.

## Step 1 - Setup

In [6]:
import logging
import os
import random
from pathlib import Path

from aieng.syn_data.text import (
    LLMSettings,
    OpenAICompatibleClient,
    load_implementation_dotenv,
    use_repo_root,
)
from rich.console import Console
from rich.table import Table


repo_root = use_repo_root()
load_implementation_dotenv()

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
console = Console(width=100)
print(f"Running from repository root: {repo_root}")

Running from repository root: /home/coder/cibc1-yujia/CIBC1


In [7]:
SEED = 42
random.seed(SEED)

SOURCE_DOCS_DIR = repo_root / "implementations/qa_text_generation/data/documents"
DESIGN_DIR = repo_root / "implementations/RAG_QA_design"
DATA_DIR = DESIGN_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

CHUNKS_PATH = DATA_DIR / "chunks.jsonl"
NAIVE_QA_PATH = DATA_DIR / "naive_qa.jsonl"
NAIVE_RESULTS_PATH = DATA_DIR / "naive_results.jsonl"

console.print(f"Source documents: {SOURCE_DOCS_DIR}")
console.print(f"Output data dir:  {DATA_DIR}")

Source documents: /home/coder/cibc1-yujia/CIBC1/implementations/qa_text_generation/data/documents

Output data dir:  /home/coder/cibc1-yujia/CIBC1/implementations/RAG_QA_design/data

### Model access check

The Vector proxy key only permits some models, so we send a tiny request to each candidate and see which ones answer.
Edit `CANDIDATE_MODELS` to test other names.

In [8]:
CANDIDATE_MODELS = [
    os.getenv("TEACHER_MODEL", "gpt-4o"),
    os.getenv("JUDGE_MODEL", "gemini-3.5-flash"),
    "gpt-4o-mini",
]
CANDIDATE_MODELS = list(dict.fromkeys(CANDIDATE_MODELS))


def check_model_access(model: str) -> tuple[bool, str]:
    """Send a tiny request and report whether the proxy allows the model (reasoning models need token headroom)."""
    client = OpenAICompatibleClient(LLMSettings.from_env(model_var="__UNSET__", default_model=model))
    try:
        reply = client.complete("Reply with the single word: ok", temperature=0.0, max_tokens=256)
    except Exception as exc:  # noqa: BLE001
        return False, str(exc)[:80]
    return True, reply.strip()[:40]


table = Table(show_header=True, header_style="bold magenta")
table.add_column("model")
table.add_column("allowed")
table.add_column("reply / error")

access = {}
for model in CANDIDATE_MODELS:
    ok, detail = check_model_access(model)
    access[model] = ok
    table.add_row(model, "yes" if ok else "NO", detail)

console.print(table)

┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ model            ┃ allowed ┃ reply / error                                                       ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gpt-4o           │ yes     │ Ok                                                                  │
│ gemini-3.5-flash │ yes     │ ok                                                                  │
│ gpt-4o-mini      │ NO      │ 403 Client Error: Forbidden for url:                                │
│                  │         │ https://proxy.vectorinstitute.ai/v1/chat/co                         │
└──────────────────┴─────────┴─────────────────────────────────────────────────────────────────────┘

## Step 2 - Document preparation

The source documents come from `qa_text_generation`:

- **CFPB credit card agreement**: in scope, this is what the RAG agent answers from.
- **SEC investor bulletin**: out of scope, used later for out-of-boundary questions.
- **Visa credit card agreement** and **cardholder agreement**: additional PDF policy documents.

The CFPB file is hard-wrapped at about 120 characters, so sentences are split across lines. The cleaner joins the wrapped lines, keeps list items on their own lines, and removes page numbers and repeated header/footer lines if there are any.

In [9]:
import re
from collections import Counter

from aieng.syn_data.text import list_domain_documents
from aieng.syn_data.text.documents import load_document_text
from aieng.syn_data.text.schemas import DocumentRole, DocumentSpec
import pymupdf


specs = {spec.doc_id: spec for spec in list_domain_documents("finance")}
for pdf_path, doc_id, title in (
    (SOURCE_DOCS_DIR / "Visa_Credit_Card_Agreement.pdf", "visa_credit_card_agreement", "Visa Credit Card Agreement"),
    (SOURCE_DOCS_DIR / "cardholder-agreement-en.pdf", "cardholder_agreement", "Cardholder Agreement"),
):
    specs[doc_id] = DocumentSpec(
        doc_id=doc_id,
        title=title,
        role=DocumentRole.POLICY_DENSE,
        domain="finance",
        local_path=str(pdf_path),
    )


def load_source_text(path: Path) -> str:
    if path.suffix.lower() == ".pdf":
        with pymupdf.open(path) as document:
            return "\n\n".join(page.get_text() for page in document)
    return load_document_text(path)


raw_texts = {doc_id: load_source_text(Path(spec.local_path)) for doc_id, spec in specs.items()}

table = Table(show_header=True, header_style="bold magenta")
for column in ("doc_id", "role", "chars", "words", "lines"):
    table.add_column(column)
for doc_id, text in raw_texts.items():
    table.add_row(
        doc_id,
        str(specs[doc_id].role),
        str(len(text)),
        str(len(text.split())),
        str(text.count("\n") + 1),
    )
console.print(table)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃ doc_id                     ┃ role           ┃ chars ┃ words ┃ lines ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ cfpb_credit_card_agreement │ policy_dense   │ 32027 │ 5541  │ 352   │
│ sec_investor_bulletin      │ scope_boundary │ 13795 │ 2133  │ 73    │
└────────────────────────────┴────────────────┴───────┴───────┴───────┘

In [10]:
PAGE_NUMBER_RE = re.compile(r"^\s*(?:page\s+)?\d+(?:\s+(?:of|/)\s+\d+)?\s*$", re.IGNORECASE)
LIST_ITEM_RE = re.compile(r"^(?:[a-z]\.\s|-\s|\d+\.\s)")  # "a. ", "- ", "1. "
SENTENCE_END = (".", "?", "!", ":", ";", '"', "\u201d", ")")
MAX_HEADER_LEN = 60  # a short line repeated 3+ times is treated as a header or footer


def clean_text(text: str) -> tuple[str, dict[str, int]]:
    """Join hard-wrapped lines, keep list items separate, and drop page numbers and repeated headers/footers."""
    lines = [line.strip() for line in text.split("\n")]
    counts = Counter(line for line in lines if line)
    repeated = {line for line, n in counts.items() if n >= 3 and len(line) <= MAX_HEADER_LEN}

    lengths = sorted(len(line) for line in lines if line)
    wrap_width = lengths[int(len(lengths) * 0.9)]  # the typical full-line width of the source

    stats = {"page_numbers_removed": 0, "headers_footers_removed": 0, "lines_joined": 0}
    blocks: list[str] = []
    for block in re.split(r"\n\s*\n", text):
        logical: list[str] = []
        current = ""
        last_len = 0
        for line in (raw.strip() for raw in block.split("\n")):
            if not line:
                continue
            if PAGE_NUMBER_RE.match(line):
                stats["page_numbers_removed"] += 1
                continue
            if line in repeated:
                stats["headers_footers_removed"] += 1
                continue
            if not current:
                current, last_len = line, len(line)
                continue
            starts_new_line = (
                LIST_ITEM_RE.match(line) is not None
                or (current.endswith(SENTENCE_END) and last_len < 0.85 * wrap_width)
                or (last_len < 0.6 * wrap_width and line[0].isupper())  # short line, then a capital: heading
            )
            if starts_new_line:
                logical.append(current)
                current = line
            else:
                current = f"{current} {line}"
                stats["lines_joined"] += 1
            last_len = len(line)
        if current:
            logical.append(current)
        if logical:
            blocks.append("\n".join(logical))
    return "\n\n".join(blocks), stats

In [11]:
clean_texts: dict[str, str] = {}
table = Table(show_header=True, header_style="bold magenta")
for column in ("doc_id", "chars before", "chars after", "lines joined", "page numbers", "headers/footers"):
    table.add_column(column)

for doc_id, text in raw_texts.items():
    cleaned, stats = clean_text(text)
    clean_texts[doc_id] = cleaned
    table.add_row(
        doc_id,
        str(len(text)),
        str(len(cleaned)),
        str(stats["lines_joined"]),
        str(stats["page_numbers_removed"]),
        str(stats["headers_footers_removed"]),
    )
console.print(table)

sample_doc = "cfpb_credit_card_agreement"
marker = "12. LIABILITY"
start = raw_texts[sample_doc].index(marker)
console.print("[bold]Before (section 12):[/bold]")
console.print(raw_texts[sample_doc][start : start + 450])
start = clean_texts[sample_doc].index(marker)
console.print("\n[bold]After (section 12):[/bold]")
console.print(clean_texts[sample_doc][start : start + 450])

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ doc_id              ┃ chars before ┃ chars after ┃ lines joined ┃ page numbers ┃ headers/footers ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ cfpb_credit_card_a… │ 32027        │ 32027       │ 225          │ 0            │ 0               │
│ sec_investor_bulle… │ 13795        │ 13795       │ 2            │ 0            │ 0               │
└─────────────────────┴──────────────┴─────────────┴──────────────┴──────────────┴─────────────────┘

Before (section 12):

12. LIABILITY FOR UNAUTHORIZED USE-LOST/STOLEN CARD NOTIFICATION — You agree to notify us
immediately, orally or in writing, at 630 32nd St. SE, Grand Rapids, MI 49548 or telephone (800) 
442-4757 seven (7)
days a week, 24 hours a day, of the loss, theft, or unauthorized use of your credit card. You may be
liable for the
unauthorized use of your credit card. You will not be liable for unauthorized use that occurs after 
you notify us of the loss,
t

After (section 12):

12. LIABILITY FOR UNAUTHORIZED USE-LOST/STOLEN CARD NOTIFICATION — You agree to notify us 
immediately, orally or in writing, at 630 32nd St. SE, Grand Rapids, MI 49548 or telephone (800) 
442-4757 seven (7) days a week, 24 hours a day, of the loss, theft, or unauthorized use of your 
credit card. You may be liable for the unauthorized use of your credit card. You will not be liable 
for unauthorized use that occurs after you notify us of the loss, t

## Step 3 - Chunking

This step reuses the existing `qa_text_generation` chunking, applied to the cleaned text:

- `paragraphs_from_document` splits each document on blank lines and merges pieces shorter than `min_chars` into their neighbour.
- `sample_test_paragraphs` holds out up to `DEFAULT_TEST_PARAS_PER_DOC` chunks per document, while keeping at least one development chunk.
- Chunk IDs have the form `doc_id::pNNNN`.

Nothing is trained in this project, so the two splits are **development** (diagnose failures and tune) and **test** (frozen, run once at the end).
The package's `ParagraphSplit` enum only has `train` and `test`, so in memory `ParagraphSplit.TRAIN` stands for development. The saved file uses the names `development` and `test`.

Chunks are saved to `data/chunks.jsonl`, one JSON object per line.

In [12]:
from aieng.syn_data.text import (
    DEFAULT_TEST_PARAS_PER_DOC,
    Paragraph,
    ParagraphSplit,
    load_typed_jsonl,
    save_typed_jsonl,
)
from aieng.syn_data.text.documents import paragraphs_from_document, sample_test_paragraphs


MIN_CHARS = 80  # same value the original build_paragraph_splits uses
DEVELOPMENT = ParagraphSplit.TRAIN  # in memory, TRAIN means development; the file says "development"


def chunk_to_record(chunk: Paragraph) -> dict:
    record = chunk.to_dict()
    record["split"] = "development" if chunk.split is DEVELOPMENT else "test"
    return record


def chunk_from_record(record: dict) -> Paragraph:
    split = ParagraphSplit.TEST.value if record["split"] == "test" else DEVELOPMENT.value
    return Paragraph.from_dict({**record, "split": split})

chunks: list[Paragraph] = []
for doc_id, spec in specs.items():
    doc_chunks = paragraphs_from_document(spec, clean_texts[doc_id], min_chars=MIN_CHARS)
    n_test = min(DEFAULT_TEST_PARAS_PER_DOC, max(0, len(doc_chunks) - 1))
    test_chunks, validation_chunks = sample_test_paragraphs(doc_chunks, n_test=n_test, seed=SEED)
    chunks.extend(test_chunks + validation_chunks)

chunks.sort(key=lambda chunk: (chunk.doc_id, chunk.index))

save_typed_jsonl(CHUNKS_PATH, chunks, to_dict=chunk_to_record)
console.print(f"Saved {len(chunks)} chunks to {CHUNKS_PATH}")

Saved 56 chunks to /home/coder/cibc1-yujia/CIBC1/implementations/RAG_QA_design/data/chunks.jsonl

In [13]:
# Reload from disk to confirm the saved file round-trips.
chunks = load_typed_jsonl(CHUNKS_PATH, chunk_from_record)

table = Table(show_header=True, header_style="bold magenta")
for column in ("doc_id", "chunks", "test", "development", "min chars", "mean chars", "max chars"):
    table.add_column(column)
for doc_id in specs:
    doc = [chunk for chunk in chunks if chunk.doc_id == doc_id]
    sizes = [len(chunk.text) for chunk in doc]
    table.add_row(
        doc_id,
        str(len(doc)),
        str(sum(chunk.split is ParagraphSplit.TEST for chunk in doc)),
        str(sum(chunk.split is DEVELOPMENT for chunk in doc)),
        str(min(sizes)),
        str(round(sum(sizes) / len(sizes))),
        str(max(sizes)),
    )
console.print(table)

for chunk in chunks[:2]:
    console.print(f"\n[bold]{chunk.para_id}[/bold]  role={chunk.role.value}  split={'development' if chunk.split is DEVELOPMENT else 'test'}  index={chunk.index}  chars={len(chunk.text)}")
    console.print(chunk.text)

console.print("\n[bold]Raw line in chunks.jsonl:[/bold]")
console.print(CHUNKS_PATH.read_text(encoding="utf-8").splitlines()[0][:400] + " ...")

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ doc_id                     ┃ chunks ┃ test ┃ development ┃ min chars ┃ mean chars ┃ max chars ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ cfpb_credit_card_agreement │ 35     │ 10   │ 25          │ 99        │ 913        │ 4376      │
│ sec_investor_bulletin      │ 21     │ 10   │ 11          │ 244       │ 655        │ 1581      │
└────────────────────────────┴────────┴──────┴─────────────┴───────────┴────────────┴───────────┘

cfpb_credit_card_agreement::p0000  role=policy_dense  split=development  index=0  chars=755

VISA CARD
CONSUMER CREDIT CARD AGREEMENT
In this Agreement, “Agreement” means this Consumer Credit Card Agreement. “Disclosure” means the 
Credit Card Account Opening Disclosure. The Account Opening Disclosure is incorporated into this 
Consumer Credit Card Agreement and is part of the Agreement. In this Agreement the words "you," 
“your,” and "yours" mean each and all of those who agree to be bound by this Agreement; "card" means
the Visa credit card and any duplicates, renewals, or substitutions the Credit Union issues to you; 
"account" means your Visa credit card line-of-credit account with the Credit Union; and "Credit 
Union" means the Credit Union whose name appears on this Agreement or anyone to whom the Credit 
Union transfers this Agreement.

cfpb_credit_card_agreement::p0001  role=policy_dense  split=development  index=1  chars=3284

1. USING YOUR ACCOUNT — If you are approved for an account, the Credit Union will establish a line 
of credit for you. You agree that your credit limit is the maximum amount (purchases, cash advances,
finance charges, plus "other charges") which you will have outstanding on your account at any time. 
Unless disclosed otherwise, the Credit Union will not allow advances over the credit limit. If the 
Credit Union has a program whereby it allows payment of advances that exceed your credit limit, 
subject to a fee, the Credit Union will provide you with notice, either orally, in writing, or 
electronically (notwithstanding the requirements of the paragraph entitled “Statements and Notices”)
explaining your right to opt in to the Credit Union’s program whereby it will honor advance requests
over the credit limit. In the event you opt in to such a program, you agree to the terms of such a 
program. You may request an increase in your credit limit only by a method acceptable to the Credit 
Union. The Credit Union may increase or decrease your credit limit, refuse to make an advance and/or
terminate your account at any time for any reason not prohibited by law. If you are permitted to 
obtain cash advances on your account, we may, from time to time, issue convenience checks to you 
that may be drawn on your account. Convenience checks may not be used to make a payment on your 
account balance. If you use a convenience check, it will be posted to your account as a cash 
advance. We reserve the right to refuse to pay a convenience check drawn on your account for any 
reason and such refusal shall not constitute wrongful dishonor.
You may request that we stop the payment of a convenience check drawn on your account. You agree to 
pay any fee imposed to stop a payment on a convenience check issued on your account. You may make a 
stop payment request orally, if permitted, or in writing. Your request must be made with sufficient 
time in advance of the presentment of the check for payment to give us a reasonable opportunity to 
act on your request. In addition, your request must accurately describe the check including the 
exact account number, the payee, any check number that may be applicable, and the exact amount of 
the check. If permitted, you may make a stop payment request orally but such a request will expire 
after 14 days unless you confirm your request in writing within that time. Written stop payment 
orders are effective only for six months and may be renewed for additional six month periods by 
requesting in writing that the stop payment order be renewed. We are not required to notify you when
a stop payment order expires. If we re-credit your account after paying a check or draft over a 
valid and timely stop payment order, you agree to sign a statement describing the dispute with the 
payee, to assign to us all of your rights against the payee or other holders of the check or draft 
and to assist us in any legal action. You agree to indemnify and hold us harmless from all costs and
expenses, including attorney's fees, damages, or claims, related to our honoring your stop payment 
request or in failing to stop payment of an item as a result of incorrect information provided to us
or the giving of inadequate time to act upon a stop payment request.

Raw line in chunks.jsonl:

{"doc_id": "cfpb_credit_card_agreement", "para_id": "cfpb_credit_card_agreement::p0000", "text": 
"VISA CARD\nCONSUMER CREDIT CARD AGREEMENT\nIn this Agreement, “Agreement” means this Consumer 
Credit Card Agreement. “Disclosure” means the Credit Card Account Opening Disclosure. The Account 
Opening Disclosure is incorporated into this Consumer Credit Card Agreement and is part of the 
Agreement. In t ...

## Step 4 - Embedding and indexing (semantic retrieval)

Chunks are embedded with `sentence-transformers` (`all-MiniLM-L6-v2`, 384 dimensions, normalized) and searched by cosine similarity.

- **Indexed:** all CFPB chunks, development and test together. The split only decides which chunks questions are generated from, so test questions can still find their source chunk.
- **Not indexed:** the SEC chunks. They are the source of out-of-scope questions, which should have nothing relevant to retrieve.
- **Long chunks:** the model reads at most 256 tokens, and 5 of the 35 CFPB chunks are longer. Each chunk is split into overlapping windows of at most 200 tokens, each window is embedded, and a chunk is scored by its best window. Results are whole chunks with the same IDs, so the ground truth for retrieval metrics does not change.
- **Saved:** the window vectors and the window-to-chunk map go to `data/index/`, so later runs load them instead of re-embedding.

In [14]:
import hashlib
import json

import numpy as np
import torch
from aieng.syn_data.text.schemas import DocumentRole
from sentence_transformers import SentenceTransformer


EMBED_MODEL = "all-MiniLM-L6-v2"
WINDOW_TOKENS = 200  # the model reads at most 256 tokens
WINDOW_STRIDE = 160  # 40 tokens of overlap between neighbouring windows
INDEX_DIR = DATA_DIR / "index"

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer(EMBED_MODEL, device=device)
console.print(f"Embedding model {EMBED_MODEL} on {device}, max_seq_length={embedder.max_seq_length}")


def split_into_windows(text: str) -> list[str]:
    """Split text into overlapping windows of at most WINDOW_TOKENS tokens, cut on token boundaries."""
    encoded = embedder.tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    offsets = encoded["offset_mapping"]
    if len(offsets) <= WINDOW_TOKENS:
        return [text]
    windows = []
    for start in range(0, len(offsets), WINDOW_STRIDE):
        end = min(start + WINDOW_TOKENS, len(offsets))
        windows.append(text[offsets[start][0] : offsets[end - 1][1]])
        if end == len(offsets):
            break
    return windows


def index_key(chunks: list[Paragraph]) -> str:
    """Fingerprint of everything the vectors depend on, so a stale saved index is never reused."""
    payload = [EMBED_MODEL, WINDOW_TOKENS, WINDOW_STRIDE] + [(chunk.para_id, chunk.text) for chunk in chunks]
    return hashlib.sha256(json.dumps(payload).encode()).hexdigest()


class SemanticIndex:
    """Window embeddings for a list of chunks. A chunk is scored by its best-matching window."""

    def __init__(self, chunks: list[Paragraph], vectors: np.ndarray, window_chunk: list[int]) -> None:
        self.chunks = list(chunks)
        self.vectors = vectors  # (n_windows, dim), L2-normalized
        self.window_chunk = np.array(window_chunk)  # window row -> chunk position

    @classmethod
    def build(cls, chunks: list[Paragraph]) -> "SemanticIndex":
        window_texts: list[str] = []
        window_chunk: list[int] = []
        for position, chunk in enumerate(chunks):
            for window in split_into_windows(chunk.text):
                window_texts.append(window)
                window_chunk.append(position)
        vectors = embedder.encode(window_texts, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
        return cls(chunks, vectors, window_chunk)

    def save(self, directory: Path) -> None:
        directory.mkdir(parents=True, exist_ok=True)
        np.save(directory / "vectors.npy", self.vectors)
        meta = {"key": index_key(self.chunks), "model": EMBED_MODEL, "window_chunk": self.window_chunk.tolist(), "chunk_ids": [chunk.para_id for chunk in self.chunks]}
        (directory / "meta.json").write_text(json.dumps(meta), encoding="utf-8")

    @classmethod
    def load(cls, directory: Path, chunks: list[Paragraph]) -> "SemanticIndex | None":
        """Load a saved index if it matches these chunks and settings, otherwise return None."""
        meta_path = directory / "meta.json"
        if not meta_path.exists():
            return None
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if meta["key"] != index_key(chunks):
            return None
        return cls(chunks, np.load(directory / "vectors.npy"), meta["window_chunk"])

    def search(self, query: str, top_k: int = 3) -> list[tuple[float, Paragraph]]:
        """Return the top_k (score, chunk) pairs, best first. Score is cosine similarity of the best window."""
        query_vector = embedder.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
        window_scores = self.vectors @ query_vector
        chunk_scores = np.full(len(self.chunks), -1.0)
        np.maximum.at(chunk_scores, self.window_chunk, window_scores)
        order = np.argsort(-chunk_scores, kind="stable")[:top_k]
        return [(float(chunk_scores[position]), self.chunks[position]) for position in order]


indexed_chunks = [chunk for chunk in chunks if chunk.role is DocumentRole.POLICY_DENSE]
index = SemanticIndex.load(INDEX_DIR, indexed_chunks)
if index is None:
    index = SemanticIndex.build(indexed_chunks)
    index.save(INDEX_DIR)
    source = "built and saved"
else:
    source = "loaded from disk"
console.print(f"Index {source}: {len(indexed_chunks)} chunks, {len(index.vectors)} windows, dim {index.vectors.shape[1]}")

/home/coder/cibc1-yujia/CIBC1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-28 18:15:06,871 INFO httpx: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-10-28 18:15:06,923 INFO httpx: HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-10-28 18:15:06,976 INFO httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-28 18:15:06,996 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2F1110a

Embedding model all-MiniLM-L6-v2 on cuda, max_seq_length=256

Index loaded from disk: 35 chunks, 55 windows, dim 384

In [15]:
SANITY_QUERIES = [
    "What is the late payment fee?",
    "When will I not be charged a finance charge on new purchases?",
    "How do I dispute a billing error on my statement?",
    "What happens if I am in default?",
    "How should I protect my online investment account from fraud?",
]

# A saved index must give the same results as a freshly built one.
fresh = SemanticIndex.build(indexed_chunks)
reloaded = SemanticIndex.load(INDEX_DIR, indexed_chunks)
assert reloaded is not None
for query in SANITY_QUERIES:
    assert [c.para_id for _, c in fresh.search(query)] == [c.para_id for _, c in reloaded.search(query)], query
console.print("Saved index matches a fresh build on all sanity queries.")

table = Table(show_header=True, header_style="bold magenta")
for column in ("query", "rank", "chunk", "split", "score", "preview"):
    table.add_column(column)
for query in SANITY_QUERIES:
    for rank, (score, chunk) in enumerate(index.search(query, top_k=3), start=1):
        table.add_row(
            query if rank == 1 else "",
            str(rank),
            chunk.para_id.split("::")[1],
            "development" if chunk.split is DEVELOPMENT else "test",
            f"{score:.3f}",
            chunk.text[:60].replace("\n", " "),
        )
console.print(table)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (664 > 256). Running this sequence through the model will result in indexing errors


Saved index matches a fresh build on all sanity queries.

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ query                        ┃ rank ┃ chunk ┃ split       ┃ score ┃ preview                      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ What is the late payment     │ 1    │ p0007 │ development │ 0.628 │ 7. FEES — In addition to the │
│ fee?                         │      │       │             │       │ periodic rate, the following │
│                              │      │       │             │       │ ad                           │
│                              │ 2    │ p0029 │ test        │ 0.442 │ 29. THE FOLLOWING IS         │
│                              │      │       │             │       │ REQUIRED BY VERMONT LAW —    │
│                              │      │       │             │       │ NOTICE TO CO-                │
│                              │ 3    │ p0031 │ test        │ 0.434 │ 1. Within 30 days of         │
│                              │      │       │             │       │ receiving your letter, we    │
│                              │      │       │             │       │ must tell you                │
│ When will I not be charged a │ 1    │ p0005 │ test        │ 0.738 │ 5. FINANCE CHARGES — New     │
│ finance charge on new        │      │       │             │       │ purchases posted to your     │
│ purchases?                   │      │       │             │       │ account du                   │
│                              │ 2    │ p0007 │ development │ 0.621 │ 7. FEES — In addition to the │
│                              │      │       │             │       │ periodic rate, the following │
│                              │      │       │             │       │ ad                           │
│                              │ 3    │ p0004 │ development │ 0.486 │ 4. PERIODIC RATES — The      │
│                              │      │       │             │       │ periodic rates applicable to │
│                              │      │       │             │       │ purchas                      │
│ How do I dispute a billing   │ 1    │ p0029 │ test        │ 0.662 │ 29. THE FOLLOWING IS         │
│ error on my statement?       │      │       │             │       │ REQUIRED BY VERMONT LAW —    │
│                              │      │       │             │       │ NOTICE TO CO-                │
│                              │ 2    │ p0031 │ test        │ 0.604 │ 1. Within 30 days of         │
│                              │      │       │             │       │ receiving your letter, we    │
│                              │      │       │             │       │ must tell you                │
│                              │ 3    │ p0032 │ development │ 0.548 │ After we finish our          │
│                              │      │       │             │       │ investigation, one of two    │
│                              │      │       │             │       │ things will ha               │
│ What happens if I am in      │ 1    │ p0011 │ development │ 0.635 │ 11. DEFAULT — You will be in │
│ default?                     │      │       │             │       │ default if you fail to make  │
│                              │      │       │             │       │ any                          │
│                              │ 2    │ p0013 │ development │ 0.270 │ 13. CHANGING OR TERMINATING  │
│                              │      │       │             │       │ YOUR ACCOUNT — As permitted  │
│                              │      │       │             │       │ by l                         │
│                              │ 3    │ p0014 │ development │ 0.246 │ 14. CHANGING OR TERMINATING  │
│                              │      │       │             │       │ AUTHORIZED USERS — Upon your │
│                              │      │       │             │       │ req                          │
│ How should I protect my      │ 1    │ p0015 │ development │

## Step 5 - Naive RAG

One answering function: embed the question, retrieve the top 5 chunks from the semantic index, and answer from them only. This is the system under test.

Model roles (different families so the judge does not favour its own answers):

| Role | Model |
|---|---|
| Question generator (Step 6) | `gpt-4o` |
| Answerer (the RAG agent) | `gemini-3.5-flash` |
| Judge (Step 7) | `gpt-4o` |

In [16]:
from dataclasses import dataclass, field


def make_client(model_var: str, default_model: str) -> OpenAICompatibleClient:
    return OpenAICompatibleClient(LLMSettings.from_env(model_var=model_var, default_model=default_model))


generator_client = make_client("RAG_GENERATOR_MODEL", "gpt-4o")
answer_client = make_client("RAG_ANSWER_MODEL", "gemini-3.5-flash")
judge_client = make_client("RAG_JUDGE_MODEL", "gpt-4o")
for role, client in ("generator", generator_client), ("answerer", answer_client), ("judge", judge_client):
    console.print(f"{role:10s} {client.settings.model}")

TOP_K = 5
ANSWER_MAX_TOKENS = 4096  # reasoning models spend 300-2000 tokens thinking before the answer; 1024 truncated answers
REFUSAL_TEXT = "Not specified in the agreement."
# Both replies the answer prompt uses for questions the agreement cannot answer.
REFUSAL_STARTS = ("not specified in the agreement", "please refer to the protection section")

ANSWER_SYSTEM_PROMPT = """
you are a credit card agreement question answering assistant. you must answer the question only based on the provided numbered context below.

# Output Rules:
* Always answer based on the provided context.
* For protection-related questions, especially regarding online investment protection, banking account privacy, or fraud, reply with: "Please refer to the protection section of your online investment for guidance."
* Do not use outside knowledge or guess.
* End with a line of the form 'Sources: <chunk ids you used>'.
* If the answer is not in the provided context, reply exactly: "Not specified in the agreement. Please contact your card issuer for more information at 123@gmail.com."
* If the question is protection related, especially regarding online investment protection or banking account privacy or fraud, you must reply with:
"Please refer to the protection section of your online investment for guidance."
* You can only provide guidance on credit card agreements.
* Never mention the "the document says" or "the passage suggests" in your answer.
* Be polite, and maintain a professional tone in your response.
"""

@dataclass
class RagResult:
    question: str
    answer: str
    retrieved_ids: list[str] = field(default_factory=list)
    retrieved_scores: list[float] = field(default_factory=list)
    model: str = ""

    @property
    def refused(self) -> bool:
        return self.answer.strip().lower().startswith(REFUSAL_STARTS)


def format_context(context_chunks: list[Paragraph]) -> str:
    return "\n\n".join(f"[{chunk.para_id.split('::')[1]}]\n{chunk.text}" for chunk in context_chunks)


def is_complete(answer: str) -> bool:
    """A finished answer ends with a Sources line, or is a refusal. Anything else was cut off."""
    text = answer.strip().lower()
    return text.startswith(REFUSAL_STARTS) or text.splitlines()[-1].startswith("sources") if text else False


def rag_answer(question: str, *, top_k: int = TOP_K) -> RagResult:
    """Retrieve the top-k chunks for the question and answer from them."""
    retrieved = index.search(question, top_k=top_k)
    prompt = f"Context passages:\n\n{format_context([chunk for _, chunk in retrieved])}\n\nQuestion: {question}"
    for _ in range(3):  # the proxy sometimes returns a cut-off answer, so ask again
        answer = answer_client.complete(prompt, system=ANSWER_SYSTEM_PROMPT, temperature=0.0, max_tokens=ANSWER_MAX_TOKENS)
        if is_complete(answer):
            break
    return RagResult(
        question=question,
        answer=answer.strip(),
        retrieved_ids=[chunk.para_id for _, chunk in retrieved],
        retrieved_scores=[round(score, 4) for score, _ in retrieved],
        model=answer_client.settings.model,
    )

generator  gpt-4o

answerer   gemini-3.5-flash

judge      gpt-4o

In [17]:
for query in SANITY_QUERIES:
    result = rag_answer(query)
    ids = ", ".join(f"{rid.split('::')[1]} ({score:.2f})" for rid, score in zip(result.retrieved_ids, result.retrieved_scores, strict=True))
    console.print(f"[bold]Q:[/bold] {query}")
    console.print(f"[bold]Retrieved:[/bold] {ids}")
    console.print(f"[bold]A:[/bold] {result.answer}\n")

Q: What is the late payment fee?

Retrieved: p0007 (0.63), p0029 (0.44), p0031 (0.43), p0032 (0.42), p0027 (0.41)

A: If your account is subject to a Late Payment Fee, the fee will be charged to your account when 
you do not make the required minimum payment by or within the number of days of the statement 
Payment Due Date set forth on the Disclosure accompanying your Agreement. The specific fee amounts 
are set forth on the Disclosure accompanying the Agreement.

Sources: p0007

Q: When will I not be charged a finance charge on new purchases?

Retrieved: p0005 (0.74), p0007 (0.62), p0004 (0.49), p0011 (0.47), p0008 (0.44)

...

Retrieved: p0011 (0.64), p0013 (0.27), p0014 (0.25), p0004 (0.24), p0009 (0.24)

A: When you are in default, the Credit Union has the right to demand immediate payment of your full 
account balance without giving you notice. If immediate payment is demanded, you agree to continue 
paying finance charges at the periodic rate charged before default until what you owe has been paid.
Additionally, any shares given as security for your account may be applied toward what you owe.

Sources: p0011

Q: How should I protect my online investment account from fraud?

Retrieved: p0015 (0.36), p0034 (0.36), p0002 (0.35), p0025 (0.34), p0010 (0.30)

A: Please refer to the protection section of your online investment for guidance.

Sources:

## Step 6 - Naive Q&A generation

The generator (`gpt-4o`) gets one chunk and writes questions with gold answers.

- **In scope:** 3 questions for each of the 25 development CFPB chunks (the 10 test chunks are not used for generation). The prompt asks for a mix of simple lookups and questions that need several facts, in customer wording, with the supporting `evidence` copied from the chunk.
- **Out of scope:** 1 question for each of the 11 development SEC chunks, written as a credit card customer asking something the agreement cannot answer. The gold answer is fixed to the refusal text. This gives roughly 13% out-of-scope questions (the README target is 15-25%, so we may raise this to 2 per chunk).
- **Checks:** evidence must appear in the chunk, then the package's heuristic filters run. No judge or support check yet.
- **Saved:** `data/naive_qa.jsonl`, with the source chunk ID, split, and prompt version on every item.

The prompts are in the next cell so they are easy to edit. Change `PROMPT_VERSION` when you change them.

In [18]:
PROMPT_VERSION = "naive-v1"
IN_SCOPE_N = 3  # questions per CFPB chunk
OUT_OF_SCOPE_N = 1  # questions per SEC chunk
GEN_TEMPERATURE = 0.7

GENERATOR_SYSTEM = "You write exam questions for evaluating a document question answering system."

IN_SCOPE_PROMPT = """You are writing exam questions about a consumer credit card agreement.
Read the passage and write {n} questions that a customer could ask.

Rules:
- Every question must be fully answerable from this passage alone.
- Write the way a real customer would. Do not copy phrases from the passage, and do not mention "the passage" or "the document".
- Mix difficulty: at least one simple lookup, and at least one question that needs two or more facts, conditions or exceptions from the passage.
- The gold answer must be complete and use only facts stated in the passage.
- "evidence" must be the exact sentence or sentences copied from the passage that support the answer.

Return JSON: {"questions": [{"question": "...", "gold_answer": "...", "evidence": "..."}]}

Passage:
{passage}"""

OUT_OF_SCOPE_PROMPT = """You are writing exam questions to test whether a credit card assistant knows its limits.
The assistant only answers questions about a consumer credit card agreement. The passage below comes from a different document (investor education material).
Write {n} question that a credit card customer might plausibly ask, about the topic of the passage, that a credit card agreement could not answer.

Rules:
- The question must be about the topic of the passage (for example investing, brokerage accounts, online account fraud), phrased as a customer asking their credit card provider.
- Write the way a real customer would. Do not mention "the passage" or "the document".
- Do not include an answer.

Return JSON: {"questions": [{"question": "..."}]}

Passage:
{passage}"""


def build_generation_prompt(chunk: Paragraph) -> str:
    in_scope = chunk.role is DocumentRole.POLICY_DENSE
    template = IN_SCOPE_PROMPT if in_scope else OUT_OF_SCOPE_PROMPT
    n = IN_SCOPE_N if in_scope else OUT_OF_SCOPE_N
    return template.replace("{n}", str(n)).replace("{passage}", chunk.text)


console.print(build_generation_prompt(indexed_chunks[3])[:1400])

You are writing exam questions about a consumer credit card agreement.
Read the passage and write 3 questions that a customer could ask.

Rules:
- Every question must be fully answerable from this passage alone.
- Write the way a real customer would. Do not copy phrases from the passage, and do not mention "the
passage" or "the document".
- Mix difficulty: at least one simple lookup, and at least one question that needs two or more 
facts, conditions or exceptions from the passage.
- The gold answer must be complete and use only facts stated in the passage.
- "evidence" must be the exact sentence or sentences copied from the passage that support the 
answer.

Return JSON: {"questions": [{"question": "...", "gold_answer": "...", "evidence": "..."}]}

Passage:
3. PROMISE TO PAY — You promise to pay all charges (purchases, cash advances, balance transfers, use
of convenience checks or any other charge) made to your account by you or anyone you authorize to 
use your account. You also promise to pay all finance charges and other charges added to your 
account under the terms of this Agreement or another agreement you made with the Credit Union. If 
this is a joint account, the paragraph on JOINT ACCOUNTS also applies to your account.

In [19]:
from concurrent.futures import ThreadPoolExecutor

from aieng.syn_data.text import QASample


def split_label(chunk: Paragraph) -> str:
    return "development" if chunk.split is DEVELOPMENT else "test"


def generate_for_chunk(chunk: Paragraph) -> list[QASample]:
    in_scope = chunk.role is DocumentRole.POLICY_DENSE
    payload = generator_client.complete_json(
        build_generation_prompt(chunk), system=GENERATOR_SYSTEM, temperature=GEN_TEMPERATURE, max_tokens=2048,
    )
    samples = []
    for k, item in enumerate(payload.get("questions", [])):
        question = str(item.get("question", "")).strip()
        gold = str(item.get("gold_answer", "")).strip() if in_scope else REFUSAL_TEXT
        samples.append(
            QASample(
                id=f"naive-{chunk.para_id}-{k}",
                question=question,
                gold_answer=gold,
                doc_id=chunk.doc_id,
                para_id=chunk.para_id,
                context=chunk.text,
                role=chunk.role,
                split=chunk.split,
                metadata={
                    "in_scope": in_scope,
                    "source_chunk_id": chunk.para_id if in_scope else None,
                    "evidence": str(item.get("evidence", "")).strip() if in_scope else "",
                    "split_name": split_label(chunk),
                    "prompt_version": PROMPT_VERSION,
                    "generator_model": generator_client.settings.model,
                },
            ),
        )
    return samples


def safe_generate(chunk: Paragraph) -> tuple[list[QASample], str | None]:
    for attempt in range(2):  # one retry, e.g. for malformed JSON
        try:
            return generate_for_chunk(chunk), None
        except Exception as exc:  # noqa: BLE001
            error = f"{type(exc).__name__}: {exc}"[:150]
    return [], error


chunks_to_generate = [chunk for chunk in chunks if chunk.split is DEVELOPMENT]  # test chunks stay untouched
with ThreadPoolExecutor(max_workers=8) as pool:
    outcomes = list(pool.map(safe_generate, chunks_to_generate))

raw_samples = [sample for samples, _ in outcomes for sample in samples]
failures = [(chunk.para_id, error) for chunk, (_, error) in zip(chunks_to_generate, outcomes, strict=True) if error]
console.print(f"Generated {len(raw_samples)} questions from {len(chunks_to_generate)} chunks; {len(failures)} chunk(s) failed")
for para_id, error in failures:
    console.print(f"  FAILED {para_id}: {error}")

Generated 86 questions from 36 chunks; 0 chunk(s) failed

In [20]:

from aieng.syn_data.text.quality import apply_heuristic_filters, summarize_heuristic_rejections


def normalize_for_match(text: str) -> str:
    text = text.replace("\u201c", '"').replace("\u201d", '"').replace("\u2018", "'").replace("\u2019", "'")
    return " ".join(text.lower().split())


# 1. In-scope evidence must be copied from the source chunk.
grounded, bad_evidence = [], []
for sample in raw_samples:
    if sample.metadata["in_scope"] and normalize_for_match(sample.metadata["evidence"]) not in normalize_for_match(sample.context):
        bad_evidence.append(sample)
    else:
        grounded.append(sample)

# 2. Cheap heuristic filters from the package (empty, length, duplicates, answer in question, leaked prompt).
naive_samples, rejected = apply_heuristic_filters(grounded)

console.print(f"raw {len(raw_samples)} -> evidence check removed {len(bad_evidence)} -> heuristics removed {len(rejected)} -> kept {len(naive_samples)}")
console.print(f"heuristic rejections: {summarize_heuristic_rejections(rejected)}")


def sample_to_record(sample: QASample) -> dict:
    record = sample.to_dict()
    record["split"] = sample.metadata["split_name"]
    return record


save_typed_jsonl(NAIVE_QA_PATH, naive_samples, to_dict=sample_to_record)
console.print(f"Saved {len(naive_samples)} questions to {NAIVE_QA_PATH}")

table = Table(show_header=True, header_style="bold magenta")
for column in ("split", "in scope", "out of scope", "total"):
    table.add_column(column)
counts = Counter((s.metadata["split_name"], s.metadata["in_scope"]) for s in naive_samples)
for split_name in ("development", "test"):
    inside, outside = counts[(split_name, True)], counts[(split_name, False)]
    table.add_row(split_name, str(inside), str(outside), str(inside + outside))
console.print(table)

for sample in naive_samples[:3] + [s for s in naive_samples if not s.metadata["in_scope"]][:2]:
    console.print(f"\n[bold]{sample.id}[/bold]  in_scope={sample.metadata['in_scope']}  split={sample.metadata['split_name']}")
    console.print(f"Q: {sample.question}")
    console.print(f"A: {sample.gold_answer}")

raw 86 -> evidence check removed 10 -> heuristics removed 0 -> kept 76

heuristic rejections: {}

Saved 76 questions to 
/home/coder/cibc1-yujia/CIBC1/implementations/RAG_QA_design/data/naive_qa.jsonl

┏━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━┓
┃ split       ┃ in scope ┃ out of scope ┃ total ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━┩
│ development │ 65       │ 11           │ 76    │
│ test        │ 0        │ 0            │ 0     │
└─────────────┴──────────┴──────────────┴───────┘

naive-cfpb_credit_card_agreement::p0000-1  in_scope=True  split=development

...

Q: How can I protect my online investment accounts from fraud?

A: Not specified in the agreement.

naive-sec_investor_bulletin::p0001-0  in_scope=False  split=development

Q: Can I use a strong passphrase for my investment account to enhance security, and how do I set 
that up?

A: Not specified in the agreement.

## Step 7 - Evaluate the RAG

The RAG answers every development question, then two evaluators score it.

**Ranking metrics** (in-scope questions only). The gold chunk is the chunk the question was written from.

- **Recall@3 / Recall@5:** share of questions whose gold chunk is in the top 3 / top 5 retrieved.
- **MRR:** average of 1 / rank of the gold chunk (0 if it was not retrieved).

**Quality judge** (`gpt-4o`). It sees the RAG system prompt (its instructions and prescribed replies), the question, the RAG answer and the retrieved chunks. It does not see the gold answer. For each of the three criteria it writes its reasoning first, then a true/false flag. Every answer is judged, because the protection rule applies to out-of-scope questions. Factuality and citation accuracy are reported on in-scope answers only.

- **Factuality:** every claim in the answer is supported by the retrieved chunks.
- **Citation accuracy:** the chunk IDs on the `Sources:` line are retrieved chunks that really support the answer.
- **Instruction following:** the answer obeys all six rules below. The judge also lists the violated rule numbers.

| Rule | Requirement |
|---|---|
| R1 | Protection-related questions (online investment protection, banking account privacy, fraud) get exactly: "Please refer to the protection section of your online investment for guidance." |
| R2 | An answer built from the context ends with a `Sources: <chunk ids you used>` line (the two exact replies are exempt) |
| R3 | If the answer is not in the context, reply exactly: "Not specified in the agreement. Please contact your card issuer for more information at 123@gmail.com." |
| R4 | Guidance on credit card agreements only |
| R5 | Never say "the document says" or "the passage suggests" |
| R6 | Polite, professional tone |

Also reported: ROUGE-L against the gold answer, and the refusal rate on the out-of-scope questions (a plain text check, no judge).

In [21]:
EVAL_SPLIT = "development"
eval_samples = [s for s in naive_samples if s.metadata["split_name"] == EVAL_SPLIT]


def safe_answer(sample: QASample) -> tuple[RagResult | None, str | None]:
    error = None
    for _ in range(2):
        try:
            return rag_answer(sample.question), None
        except Exception as exc:  # noqa: BLE001
            error = f"{type(exc).__name__}: {exc}"[:150]
    return None, error


with ThreadPoolExecutor(max_workers=8) as pool:
    answered = list(pool.map(safe_answer, eval_samples))

rows = []
for sample, (result, error) in zip(eval_samples, answered, strict=True):
    if result is None:
        console.print(f"  FAILED {sample.id}: {error}")
        continue
    rows.append(
        {
            "id": sample.id,
            "question": sample.question,
            "in_scope": sample.metadata["in_scope"],
            "gold_chunk": sample.metadata["source_chunk_id"],
            "gold_answer": sample.gold_answer,
            "answer": result.answer,
            "refused": result.refused,
            "retrieved_ids": result.retrieved_ids,
            "retrieved_scores": result.retrieved_scores,
        },
    )
console.print(f"Answered {len(rows)} of {len(eval_samples)} questions ({sum(r['in_scope'] for r in rows)} in scope)")

Answered 76 of 76 questions (65 in scope)

In [22]:
from statistics import mean


KS = (3, 5)


def gold_rank(row: dict) -> int | None:
    """1-based position of the gold chunk in the retrieved list, or None if it was not retrieved."""
    ids = row["retrieved_ids"]
    return ids.index(row["gold_chunk"]) + 1 if row["gold_chunk"] in ids else None


def ranking_metrics(scored_rows: list[dict]) -> dict[str, float]:
    ranks = [row["gold_rank"] for row in scored_rows]
    metrics = {f"Recall@{k}": mean(1.0 if rank is not None and rank <= k else 0.0 for rank in ranks) for k in KS}
    metrics["MRR"] = mean(1 / rank if rank else 0.0 for rank in ranks)
    return metrics


in_scope_rows = [row for row in rows if row["in_scope"]]
for row in in_scope_rows:
    row["gold_rank"] = gold_rank(row)

ranking = ranking_metrics(in_scope_rows)
table = Table(show_header=True, header_style="bold magenta")
table.add_column(f"Ranking ({len(in_scope_rows)} in-scope questions)")
table.add_column("value")
for name, value in ranking.items():
    table.add_row(name, f"{value:.3f}")
console.print(table)

for row in [r for r in in_scope_rows if r["gold_rank"] is None]:
    console.print(f"[bold]Gold chunk not retrieved:[/bold] {row['gold_chunk'].split('::')[1]}  Q: {row['question']}")

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Ranking (65 in-scope questions) ┃ value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ Recall@3                        │ 0.923 │
│ Recall@5                        │ 0.985 │
│ MRR                             │ 0.816 │
└─────────────────────────────────┴───────┘

Gold chunk not retrieved: p0004  Q: What happens if I don't make a payment by the due date?

In [23]:
JUDGE_SYSTEM = "You are a strict evaluator of answers produced by a document question answering system."

JUDGE_PROMPT = """You are given the instructions a RAG system was told to follow, a question, the answer the system produced, and the document chunks it retrieved.
Each chunk starts with its ID in square brackets.
Judge the answer using ONLY the retrieved chunks and the RAG instructions. You are not given a reference answer, and you must not use outside knowledge.

RAG instructions (the system prompt the answering system was given):
<<<
{rag_instructions}
>>>

Criteria, each a true/false flag:
1. factuality: true only if every claim in the answer is supported by the retrieved chunks. False if any claim is missing from the chunks, contradicts them, or comes from outside knowledge. The fixed reply the instructions prescribe for unanswerable questions makes no claim, so it is true, unless the retrieved chunks clearly do contain the answer.
2. citation_accuracy: true only if the answer ends with the "Sources:" line the instructions require, every listed ID is among the retrieved chunks, and each listed chunk really contains the information the answer uses. False if a source is missing, invented, or does not support the answer. A prescribed refusal reply has no claims to cite, so it is true.
3. instruction_following: true only if the answer obeys ALL of these rules, which are the rules from the RAG instructions that matter:
   R1. For a protection-related question (online investment protection, banking account privacy, or fraud), the reply must be exactly: "Please refer to the protection section of your online investment for guidance."
   R2. An answer built from the context must end with a line of the form "Sources: <chunk ids you used>". The two exact replies in R1 and R3 are exempt from this rule.
   R3. If the answer is not in the retrieved chunks, the reply must be exactly: "Not specified in the agreement. Please contact your card issuer for more information at 123@gmail.com." It is a violation to use this reply when the chunks do contain the answer, and a violation to answer from outside the chunks when they do not.
   R4. The answer may only give guidance on credit card agreements.
   R5. The answer must never say "the document says", "the passage suggests" or similar wording that refers to the document or passage.
   R6. The tone must be polite and professional.
   The prescribed texts in R1 and R3 are required by the instructions, so they are never irrelevant content. Do not mark other things (format, length, completeness) as violations, because only R1-R6 count here.

For each criterion write the reasoning first, in one to three sentences that point to the evidence. For instruction_following the reasoning must say which rules were violated, if any. Then give the flag. Also list the violated rule numbers.

Return JSON with exactly these keys:
{"factuality_reasoning": "...", "factuality": true, "citation_accuracy_reasoning": "...", "citation_accuracy": true, "instruction_following_reasoning": "...", "instruction_following": true, "violated_rules": []}
violated_rules is a list of strings such as ["R2", "R5"], empty when instruction_following is true.

Question:
{question}

Retrieved chunks:
{context}

Answer:
{answer}"""

CRITERIA = ("factuality", "citation_accuracy", "instruction_following")
chunk_by_id = {chunk.para_id: chunk for chunk in indexed_chunks}


def build_judge_prompt(row: dict) -> str:
    context = format_context([chunk_by_id[chunk_id] for chunk_id in row["retrieved_ids"]])
    prompt = JUDGE_PROMPT.replace("{rag_instructions}", ANSWER_SYSTEM_PROMPT.strip())
    return prompt.replace("{question}", row["question"]).replace("{context}", context).replace("{answer}", row["answer"])


def parse_verdict(payload: dict) -> dict:
    verdict = {}
    for name in CRITERIA:
        if not isinstance(payload[name], bool):
            raise ValueError(f"{name} is not a boolean: {payload[name]!r}")
        verdict[name] = payload[name]
        verdict[f"{name}_reasoning"] = str(payload[f"{name}_reasoning"]).strip()
    violated = payload.get("violated_rules", [])
    if not isinstance(violated, list):
        raise ValueError(f"violated_rules is not a list: {violated!r}")
    verdict["violated_rules"] = [str(rule).strip().upper() for rule in violated]
    return verdict


def safe_judge(row: dict) -> tuple[dict | None, str | None]:
    error = None
    for _ in range(2):  # one retry, e.g. for malformed JSON
        try:
            payload = judge_client.complete_json(build_judge_prompt(row), system=JUDGE_SYSTEM, temperature=0.0, max_tokens=1024)
            return parse_verdict(payload), None
        except Exception as exc:  # noqa: BLE001
            error = f"{type(exc).__name__}: {exc}"[:150]
    return None, error


with ThreadPoolExecutor(max_workers=8) as pool:
    judged = list(pool.map(safe_judge, rows))  # every answer: the protection rule applies to out-of-scope questions

for row, (verdict, error) in zip(rows, judged, strict=True):
    if verdict is None:
        console.print(f"  JUDGE FAILED {row['id']}: {error}")
    row["judge"] = verdict
console.print(f"Judged {sum(row['judge'] is not None for row in rows)} of {len(rows)} answers")

Judged 76 of 76 answers

In [24]:
from rich.markup import escape
from rouge_score import rouge_scorer


rouge = rouge_scorer.RougeScorer(["rougeL"])


def strip_sources(answer: str) -> str:
    return re.sub(r"\s*Sources:.*$", "", answer, flags=re.IGNORECASE | re.DOTALL).strip()


for row in in_scope_rows:
    row["rouge_l"] = rouge.score(row["gold_answer"], strip_sources(row["answer"]))["rougeL"].fmeasure

judged_rows = [row for row in rows if row["judge"] is not None]
judged_in_scope = [row for row in judged_rows if row["in_scope"]]
out_of_scope_rows = [row for row in rows if not row["in_scope"]]


def rate(selected: list[dict], name: str) -> str:
    return f"{mean(1.0 if row['judge'][name] else 0.0 for row in selected):.3f} ({len(selected)})"


table = Table(show_header=True, header_style="bold magenta")
table.add_column("Quality judge (share true, with number of answers)")
table.add_column("in scope")
table.add_column("out of scope")
table.add_row("factuality", rate(judged_in_scope, "factuality"), "-")
table.add_row("citation_accuracy", rate(judged_in_scope, "citation_accuracy"), "-")
judged_out = [row for row in judged_rows if not row["in_scope"]]
table.add_row("instruction_following", rate(judged_in_scope, "instruction_following"), rate(judged_out, "instruction_following"))
console.print(table)

violations = Counter(rule for row in judged_rows for rule in row["judge"]["violated_rules"])
console.print(f"Instruction rule violations (R1 protection reply, R2 Sources line, R3 not-specified reply, R4 scope, R5 wording, R6 tone): {dict(sorted(violations.items())) or 'none'}")

table = Table(show_header=True, header_style="bold magenta")
table.add_column("Other metrics")
table.add_column("value")
table.add_row("ROUGE-L F (in scope, mean)", f"{mean(row['rouge_l'] for row in in_scope_rows):.3f}")
table.add_row(f"Refused, out of scope ({len(out_of_scope_rows)} questions; want 1.0)", f"{mean(1.0 if row['refused'] else 0.0 for row in out_of_scope_rows):.3f}")
table.add_row("Refused, in scope (over-refusal; want 0.0)", f"{mean(1.0 if row['refused'] else 0.0 for row in in_scope_rows):.3f}")
console.print(table)

for row in [r for r in judged_rows if not all(r["judge"][name] for name in CRITERIA)][:5]:
    console.print(f"\n[bold]{escape(row['id'].split('::')[-1])}[/bold]  in_scope={row['in_scope']}  Q: {escape(row['question'])}")
    console.print(f"A: {escape(row['answer'])}")
    for name in CRITERIA:
        console.print(f"  {name}={row['judge'][name]}: {escape(row['judge'][name + '_reasoning'])}")
    console.print(f"  violated_rules={row['judge']['violated_rules']}")

with NAIVE_RESULTS_PATH.open("w", encoding="utf-8") as handle:
    for row in rows:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
console.print(f"\nSaved {len(rows)} results to {NAIVE_RESULTS_PATH}")

2026-10-28 18:16:48,160 INFO absl: Using default tokenizer.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Quality judge (share true, with number of answers) ┃ in scope   ┃ out of scope ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ factuality                                         │ 1.000 (65) │ -            │
│ citation_accuracy                                  │ 1.000 (65) │ -            │
│ instruction_following                              │ 1.000 (65) │ 1.000 (11)   │
└────────────────────────────────────────────────────┴────────────┴──────────────┘

Instruction rule violations (R1 protection reply, R2 Sources line, R3 not-specified reply, R4 scope,
R5 wording, R6 tone): none

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Other metrics                                  ┃ value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ ROUGE-L F (in scope, mean)                     │ 0.536 │
│ Refused, out of scope (11 questions; want 1.0) │ 1.000 │
│ Refused, in scope (over-refusal; want 0.0)     │ 0.000 │
└────────────────────────────────────────────────┴───────┘

Saved 76 results to 
/home/coder/cibc1-yujia/CIBC1/implementations/RAG_QA_design/data/naive_results.jsonl

In [25]:
import pandas as pd


EXPORT_DIR = DATA_DIR / "exports"
EXPORT_DIR.mkdir(exist_ok=True)

short = lambda chunk_id: chunk_id.split("::")[1] if chunk_id else ""

answers_df = pd.DataFrame(
    {
        "id": row["id"].split("::")[1] if row["in_scope"] else row["id"],
        "in_scope": row["in_scope"],
        "question": row["question"],
        "rag_answer": row["answer"],
        "gold_answer": row["gold_answer"],
        "rouge_l": round(row["rouge_l"], 3) if "rouge_l" in row else None,
        "rag_words": len(strip_sources(row["answer"]).split()),
        "gold_words": len(row["gold_answer"].split()),
        "gold_chunk": short(row["gold_chunk"]),
        "gold_rank": row.get("gold_rank"),
        "retrieved": ", ".join(short(i) for i in row["retrieved_ids"]),
        "refused": row["refused"],
    }
    for row in rows
)

judge_df = pd.DataFrame(
    {
        "id": row["id"].split("::")[1] if row["in_scope"] else row["id"],
        "in_scope": row["in_scope"],
        "question": row["question"],
        "rag_answer": row["answer"],
        "retrieved": ", ".join(short(i) for i in row["retrieved_ids"]),
        "factuality": row["judge"]["factuality"],
        "factuality_reasoning": row["judge"]["factuality_reasoning"],
        "citation_accuracy": row["judge"]["citation_accuracy"],
        "citation_accuracy_reasoning": row["judge"]["citation_accuracy_reasoning"],
        "instruction_following": row["judge"]["instruction_following"],
        "violated_rules": ", ".join(row["judge"]["violated_rules"]),
        "instruction_following_reasoning": row["judge"]["instruction_following_reasoning"],
    }
    for row in judged_rows
)


def save_excel(frame: pd.DataFrame, path) -> None:
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        frame.to_excel(writer, index=False)
        sheet = writer.sheets["Sheet1"]
        sheet.freeze_panes = "A2"
        for column in sheet.columns:
            header = column[0].value
            wide = header in {"question", "rag_answer", "gold_answer"} or header.endswith("_reasoning")
            sheet.column_dimensions[column[0].column_letter].width = 60 if wide else 16
            for cell in column:
                cell.alignment = Alignment(wrap_text=True, vertical="top")


from openpyxl.styles import Alignment


save_excel(answers_df, EXPORT_DIR / "naive_answers.xlsx")
save_excel(judge_df, EXPORT_DIR / "naive_judge.xlsx")
console.print(f"Saved {EXPORT_DIR / 'naive_answers.xlsx'} ({len(answers_df)} rows)")
console.print(f"Saved {EXPORT_DIR / 'naive_judge.xlsx'} ({len(judge_df)} rows)")

scored = answers_df[answers_df["rouge_l"].notna()]
console.print(f"Mean words: RAG answer {scored['rag_words'].mean():.0f}, gold answer {scored['gold_words'].mean():.0f}")

Saved /home/coder/cibc1-yujia/CIBC1/implementations/RAG_QA_design/data/exports/naive_answers.xlsx 
(76 rows)

Saved /home/coder/cibc1-yujia/CIBC1/implementations/RAG_QA_design/data/exports/naive_judge.xlsx (76 
rows)

Mean words: RAG answer 51, gold answer 23